# Hidden Hop - MuSiQue


## 0. What this notebook needs from outside

In [ ]:
# ============================================================================
# Files this notebook reads (nothing else):
#   models/musique_sa_selector_ans.pt   the paper's RoBERTa paragraph selector (optional -
#                                       RETRIEVER_NAME = "bm25" needs no checkpoint)
#   paper_selector.py                   its loader and the .retrieve(query, k) wrapper
# The dataset comes from HuggingFace on first run and is cached there.
# ============================================================================
import os, sys
from pathlib import Path

HERE = os.path.abspath(os.getcwd())
if os.path.basename(HERE).lower() != "notebooks" and os.path.isdir(os.path.join(HERE, "notebooks")):
    HERE = os.path.join(HERE, "notebooks")


def find_selector_checkpoint():
    """Resolve the optional large checkpoint without assuming this repository's location."""
    name = "musique_sa_selector_ans.pt"
    override = os.getenv("MUSIQUE_SELECTOR_CKPT")
    candidates = ([Path(override).expanduser()] if override else [])
    candidates += [Path(HERE) / "models" / name]
    candidates += [parent / "models" / name for parent in Path(HERE).parents]
    return str(next((p.resolve() for p in candidates if p.is_file()), candidates[0]))


for _f in ["paper_selector.py"]:
    print(f"{_f:<34} {'found' if os.path.exists(os.path.join(HERE, _f)) else 'MISSING'}")
SELECTOR_CKPT = find_selector_checkpoint()
if os.path.isfile(SELECTOR_CKPT):
    print(f"{'MuSiQue selector checkpoint':<34} found: {SELECTOR_CKPT}")
else:
    print(f"{'MuSiQue selector checkpoint':<34} MISSING")
    print(f"Run `{sys.executable} download_musique_selector.py` from {HERE}, or set MUSIQUE_SELECTOR_CKPT.")


## 1. Install

In [ ]:
!pip install -q datasets rank_bm25 openai python-dotenv pandas numpy tqdm transformers torch gdown


## 2. API key

In [13]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
import os, getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")
print("key set")

key set


## 3. Imports

In [15]:
import collections, io, json, math, os, random, re, string, sys, threading, time
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from openai import OpenAI

# Also set in cell 2, which is optional; defining it here means every cell below
# works whether or not that check was run.
HERE = os.path.abspath(os.getcwd())
if os.path.basename(HERE).lower() != "notebooks" and os.path.isdir(os.path.join(HERE, "notebooks")):
    HERE = os.path.join(HERE, "notebooks")

client = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")
_create = client.chat.completions.create
def _via_openrouter(*a, **kw):
    if "/" not in kw["model"]:
        kw["model"] = "openai/" + kw["model"]
    return _create(*a, **kw)
client.chat.completions.create = _via_openrouter
print("imports ok")

imports ok


## 4. Config

`SEED` names the sample: two arms are only comparable when they ran on the same seed. `TOP_K` is
the one-shot arm's budget and `SOLVE_K` is the per-subquestion budget - they are separate because
the decomposing arms issue several retrievals per question.

In [16]:
# ---------------- CENTRAL CONFIG ----------------
READER_MODEL    = "gpt-4o-mini"   # the reader for every arm
TOP_K           = 5              # passages for the one-shot arm
N_SAMPLES       = None            # hop-stratified sample; None = the whole dev split (2,417)
SEED            = 71               # names the sample; change it and the arms are no longer paired
WORKERS         = 8               # threads over questions
RETRIEVER_NAME  = "selector"      # "selector" (the paper's RoBERTa ranker) | "bm25"
# Cell 2 normally defines this. Repeat the portable lookup so this configuration cell also works alone.
if "find_selector_checkpoint" not in globals():
    from pathlib import Path
    def find_selector_checkpoint():
        name = "musique_sa_selector_ans.pt"
        override = os.getenv("MUSIQUE_SELECTOR_CKPT")
        candidates = ([Path(override).expanduser()] if override else [])
        candidates += [Path(HERE) / "models" / name]
        candidates += [parent / "models" / name for parent in Path(HERE).parents]
        return str(next((p.resolve() for p in candidates if p.is_file()), candidates[0]))
SELECTOR_CKPT   = find_selector_checkpoint()
ARMS            = ["retrieve", "decompose", "split"]

print(f"reader={READER_MODEL}  k={TOP_K}  n={N_SAMPLES}  seed={SEED}  "
      f"retriever={RETRIEVER_NAME}")

reader=gpt-4o-mini  k=5  n=None  seed=71  retriever=selector


## 5. Load MuSiQue and draw the sample

The dev split is 2,417 answerable instances - 1,252 two-hop, 760 three-hop, 405 four-hop. The
sample keeps that mix, because the arms separate by hop count and an unbalanced sample would
confound the comparison with difficulty.

In [17]:
LOCAL_JSONL = None      # e.g. r"C:\data\musique_ans_v1.0_dev.jsonl" to skip the download

def _norm_instance(ex):
    """One schema, whichever field spellings the source uses."""
    paras = []
    for p in ex["paragraphs"]:
        paras.append({
            "idx":    p.get("idx", len(paras)),
            "title":  p.get("title", ""),
            "text":   p.get("paragraph_text", p.get("text", "")),
            "is_supporting": bool(p.get("is_supporting", False)),
        })
    # `serial` is what the pipeline sorts the pool by. On FC-MH it is chronological and decides
    # which of two conflicting facts wins; MuSiQue has no such order, so it is the paragraph's
    # own index - stable and unique, used only to make the pool deterministic.
    for p in paras:
        p["serial"] = p["idx"]
    return {"id": ex.get("id", ""), "question": ex["question"], "answer": ex["answer"],
            "answer_aliases": ex.get("answer_aliases", ex.get("answer_alias", [])) or [],
            "decomposition": ex.get("question_decomposition", []),
            "paragraphs": paras}


def load_musique():
    if LOCAL_JSONL:
        rows = [json.loads(l) for l in io.open(LOCAL_JSONL, encoding="utf-8")]
        return [_norm_instance(r) for r in rows]
    from datasets import load_dataset
    for name, kw in [("dgslibisey/MuSiQue", {"split": "validation"}),
                     ("bdsaglam/musique", {"name": "answerable", "split": "validation"})]:
        try:
            return [_norm_instance(ex) for ex in load_dataset(name, **kw)]
        except Exception as e:
            print(f"  {name} failed: {e}")
    raise RuntimeError("could not load MuSiQue; set LOCAL_JSONL to the official dev jsonl")


def hops(inst):
    return len(inst["decomposition"]) or 2


def stratified_sample(pool, n, seed):
    """Keep the dev split's hop mix, so a sample is not accidentally easier than the split.

    n = None means the whole split. Note that with n = len(pool) every hop is allocated in
    full, so the seed changes only the ORDER of the returned list, not its membership --
    and since questions are scored independently, the seed stops affecting any result.
    """
    if n is None or n >= len(pool):
        n = len(pool)
    by_hop = collections.defaultdict(list)
    for inst in pool:
        by_hop[hops(inst)].append(inst)
    tot = len(pool)
    exact = {h: len(v) * n / tot for h, v in by_hop.items()}
    alloc = {h: int(math.floor(v)) for h, v in exact.items()}
    for h in sorted(exact, key=lambda h: -(exact[h] - alloc[h]))[:n - sum(alloc.values())]:
        alloc[h] += 1
    rng = random.Random(seed)
    out = []
    for h in sorted(alloc):
        out += rng.sample(sorted(by_hop[h], key=lambda i: i["id"]), alloc[h])
    rng.shuffle(out)
    return out


DEV = load_musique()
QUESTIONS = stratified_sample(DEV, N_SAMPLES, SEED)
print(f"dev split: {len(DEV)} instances, hop mix "
      f"{dict(sorted(collections.Counter(hops(i) for i in DEV).items()))}")
print(f"sample:    {len(QUESTIONS)} instances, hop mix "
      f"{dict(sorted(collections.Counter(hops(i) for i in QUESTIONS).items()))}")

dev split: 2417 instances, hop mix {2: 1252, 3: 760, 4: 405}
sample:    2417 instances, hop mix {2: 1252, 3: 760, 4: 405}


## 6. Data preview - what one instance contains

In [18]:
_ex = QUESTIONS[0]
print("question :", _ex["question"])
print("answer   :", _ex["answer"], "  aliases:", _ex["answer_aliases"])
print("hops     :", hops(_ex))
print("\ngold decomposition (the reference plan - never shown to the model):")
for i, d in enumerate(_ex["decomposition"], 1):
    print(f"  {i}. {d.get('question', '')}   -> {d.get('answer', '')}")
print(f"\nparagraphs: {len(_ex['paragraphs'])} "
      f"({sum(p['is_supporting'] for p in _ex['paragraphs'])} supporting)")
for p in _ex["paragraphs"][:3]:
    print(f"  [{p['serial']:>2}]{' GOLD' if p['is_supporting'] else '    '} "
          f"{p['title']}: {p['text'][:110]}...")

question : In which years did the war with the military leader from which the term Pyrrhic victory comes occur?
answer   : 323–272 BC   aliases: []
hops     : 2

gold decomposition (the reference plan - never shown to the model):
  1. What military leader does the term Pyrrhic victory come from?   -> Pyrrhus
  2. In which years did the war with #1 occur?   -> 323–272 BC

paragraphs: 20 (2 supporting)
  [ 0]     Bert Combs: Combs rose from poverty in his native Clay County to obtain a law degree from the University of Kentucky and o...
  [ 1]     Battle of Vĩnh Yên: The Battle of Vĩnh Yên (), also called Tran Hung Dao Campaign by Vietminh, which occurred from 13 to 17 Januar...
  [ 2] GOLD Hieronymus of Cardia: He wrote a history of the Diadochi and their descendants, encompassing the period from the death of Alexander ...


## 7. Retriever - one `.retrieve(query, k)`, built per instance

MuSiQue gives each question its own 20 candidate paragraphs, so retrieval is a ranking problem
over 20 items rather than a search over a corpus. Default is the paper's RoBERTa selector, which
is what every other MuSiQue number in this project used; BM25 is the no-checkpoint fallback.

In [20]:
from rank_bm25 import BM25Okapi

class BM25Retriever:
    def __init__(self, passages):
        self.passages = passages
        self._bm25 = BM25Okapi([(p["title"] + " " + p["text"]).lower().split() for p in passages])

    def retrieve(self, query, k=5):
        s = self._bm25.get_scores(query.lower().split())
        return [(self.passages[i], float(s[i])) for i in np.argsort(s)[::-1][:k]]


if RETRIEVER_NAME == "selector":
    if not os.path.isfile(SELECTOR_CKPT):
        raise FileNotFoundError(
            "MuSiQue selector checkpoint not found"
            f"From {HERE}, run `{sys.executable} download_musique_selector.py`, "
            "or set MUSIQUE_SELECTOR_CKPT to an existing converted checkpoint."
        )
    sys.path.insert(0, HERE)
    from paper_selector import make_builder
    import torch
    _dev = "cuda" if torch.cuda.is_available() else "cpu"
    build_retriever, _sel_model, _sel_tok = make_builder(
        SELECTOR_CKPT, device=_dev,
        dtype=torch.float16 if _dev == "cuda" else torch.float32)
    print(f"paper selector loaded on {_dev}")
else:
    build_retriever = BM25Retriever
    print("BM25 over each instance's own paragraphs")


# The pipeline calls a module-global `retriever`. MuSiQue builds one per question and the run is
# threaded, so a plain global would race: this forwards to whichever retriever belongs to the
# question the current worker is solving.
_local = threading.local()


class _BoundRetriever:
    def retrieve(self, query, k=5):
        return _local.retr.retrieve(query, k)


retriever = _BoundRetriever()


def _bind(r):
    _local.retr = r


print("retriever ready")

paper selector loaded on cuda
retriever ready


## 9. The prompts

In [ ]:
# ---------------- Config for the recursive method ----------------
MAX_SPLIT_DEPTH = 2        # how deep a subquestion may be split again. 0 disables splitting.
SOLVE_K         = TOP_K    # passages retrieved per subquestion
SOLVE_K_DEEP    = 0        # optional deeper pool, retried only on a node that could not answer
SOLVE_VERBOSE   = False    # print every node as it is solved
SOLVE_RETRIES   = 3        # retries for transient API errors

ANSWER_SYS = (
    "You answer questions using ONLY the passages you are given. Answer the Question from the "
    "Passages. You may combine passages if the answer needs more than one of them. Your answer "
    "MUST be a word or phrase that appears in the passage text. If the passages truly do not "
    "contain the answer, output exactly: unknown. "
    "Never answer from your own knowledge of the world, even when you are confident and even as "
    "a fallback: an answer that is not supported by these passages is wrong here, whether or not "
    "it is true. "
    "Give a very concise answer - one word or short phrase only, with no explanation."
)

# The same prompt with the refusal removed, for a node that has to commit.
GUESS_SYS = ANSWER_SYS.replace(
    "If the passages truly do not contain the answer, output exactly: unknown. ",
    "Even when the passages are a poor match, give the closest word or phrase in them; never "
    "refuse and never answer 'unknown'. ")

# Reached only after ANSWER_SYS has said "unknown". Finds the entity that must be resolved first;
# never answers the question.
BRIDGE_SYS = (
    "You work ONLY over the passages you are given, never from your own knowledge of the world.\n"
    "A reader has already tried and failed to answer the Question from these passages. Do NOT "
    "answer the question. Your only job is to find the stepping stone.\n"
    "The passages often do not state the asked relation for the asked entity, but do state some "
    "OTHER relation of that entity, which names a NEW entity - and the asked relation is recorded "
    "for the new entity instead. They may not say which country a film was made in but may name "
    "its director; they may not give a person's birthplace but may name the band they founded; "
    "they may not say when a building opened but may name the company that built it.\n"
    "Find the ONE sentence in the passages that names that new entity, and copy it verbatim. It "
    "will normally mention an entity the question already names - that is how it connects. What "
    "makes it a stepping stone is the NEW name it introduces.\n"
    "A sentence is NOT a stepping stone if it introduces no new name, or if it is about an entity "
    "the question is not asking about.\n"
    "Output EXACTLY ONE line:\n"
    "  BRIDGE: <one sentence from the passages, copied verbatim>\n"
    "  NONE\n"
    "If no sentence in the passages names such an entity, output exactly: NONE"
)

# Turns (question, bridge sentence) into two subquestions, the second chaining on #1.
SPLIT_SYS = (
    "A question cannot be answered directly from the passages, but a bridge sentence was found. "
    "Split the question into EXACTLY TWO simpler subquestions using that bridge sentence.\n"
    "Rules:\n"
    "- Subquestion 1 must be answerable by the bridge sentence.\n"
    "- Subquestion 2 must contain the placeholder #1 (the answer of subquestion 1) and, once #1 "
    "is filled in, must answer the original question.\n"
    "- Both subquestions must be about entities named in the question or the bridge sentence you "
    "were given. Never mention an entity from the examples.\n"
    "- Output EXACTLY two lines, nothing else:\n"
    "1. <subquestion 1>\n"
    "2. <subquestion 2 containing #1>"
)

SPLIT_FEWSHOT = (
    "### EXAMPLES (format only - do NOT answer these, do NOT reuse their entities)\n"
    "Question: In which city was the author of The Ashen Ledger born?\n"
    "Bridge sentence: The Ashen Ledger was written by Ilvane Sabreth.\n"
    "1. Who is the author of The Ashen Ledger?\n"
    "2. In which city was #1 born?\n\n"
    "Question: Which country is the spouse of Marren Volkov a citizen of?\n"
    "Bridge sentence: Marren Volkov married Orrin Fennwick in 1998.\n"
    "1. Who is Marren Volkov married to?\n"
    "2. Which country is #1 a citizen of?\n"
    "### END EXAMPLES\n\n"
    "Now the REAL question. Both subquestions must be about the entities in THIS question and its "
    "bridge sentence, and nothing from the examples above.\n"
)

print(f"prompts ready: max_split_depth={MAX_SPLIT_DEPTH} solve_k={SOLVE_K} "
      f"reader={READER_MODEL}")

## 10. The decomposer


In [ ]:
DECOMP_SYS = (
    "You decompose a multi-hop question into an ordered list of single-hop subquestions.\n"
    "Rules:\n"
    "- One subquestion per line, numbered '1.', '2.', ...\n"
    "- Every subquestion must name an entity that appears in the question you were given, or "
    "refer to an earlier answer with #k (e.g. #1 = answer to subquestion 1).\n"
    "- Each subquestion must be answerable on its own once the #k references are filled in.\n"
    "- Some questions need two INDEPENDENT lookups rather than a chain. When neither fact depends "
    "on the other, ask for them in SEPARATE subquestions with no #k placeholder, then combine "
    "them in a later subquestion that references both (for example '... #1 ... #2 ...'). Do not "
    "force a parallel question into a single chain.\n"
    "- If the question is already single-hop, output exactly one line.\n"
    "Output ONLY the numbered list, nothing else."
)

DECOMP_FEWSHOT = (
    "### EXAMPLES (format only - do NOT answer these)\n"
    "Question: In which city was the author of The Ashen Ledger born?\n"
    "1. Who is the author of The Ashen Ledger?\n"
    "2. In which city was #1 born?\n\n"
    "Question: Which country is the spouse of Marren Volkov a citizen of?\n"
    "1. Who is Marren Volkov married to?\n"
    "2. Which country is #1 a citizen of?\n\n"
    "Question: When was the first establishment that McDonaldization is named after, open in the "
    "country Horndean is located?\n"
    "1. What is McDonaldization named after?\n"
    "2. Which country is Horndean located in?\n"
    "3. When did the first #1 open in #2?\n\n"
    "Question: What language does Orrin Fennwick speak?\n"
    "1. What language does Orrin Fennwick speak?\n"
    "### END EXAMPLES\n\n"
    "Now decompose the REAL question below. Decompose only that question - do not answer it and "
    "do not reproduce any example above.\n"
)


def decompose_question(question):
    """Break a question into ordered subquestion templates (with #k placeholders).
    Returns (subqs, n_hops)."""
    resp = client.chat.completions.create(
        model=READER_MODEL,
        messages=[{"role": "system", "content": DECOMP_SYS},
                  {"role": "user",   "content": DECOMP_FEWSHOT + "\nQuestion: " + question + "\n"}],
        max_tokens=256, temperature=0.0, seed=42,
    )
    subqs = []
    for line in resp.choices[0].message.content.strip().split("\n"):
        m = re.match(r"^\s*\d+[.)]\s*(.+)$", line.strip())
        if m:
            subqs.append(m.group(1).strip())
    if not subqs:                 # fallback: treat as single-hop
        subqs = [question]
    return subqs, len(subqs)


def _fill(template, prev_answers):
    """Substitute #k with the k-th resolved answer (1-indexed)."""
    def repl(m):
        idx = int(m.group(1)) - 1
        return prev_answers[idx] if 0 <= idx < len(prev_answers) else m.group(0)
    return re.sub(r"#(\d+)", repl, template)


print("decomposer ready")

## 11. The method

In [8]:
# ---------------- 13. the pipeline ----------------
# solve_sequence <-> solve are mutually recursive; that pair is the whole flow chart.
import re, time

def _chat(system, user, max_tokens=96):
    """One reader call. Transient API errors are retried so they cannot become wrong answers."""
    for attempt in range(SOLVE_RETRIES):
        try:
            r = client.chat.completions.create(
                model=READER_MODEL,
                messages=[{"role": "system", "content": system},
                          {"role": "user",   "content": user}],
                max_tokens=max_tokens, temperature=0.0, seed=42,
            )
            return r.choices[0].message.content.strip()
        except Exception:
            if attempt == SOLVE_RETRIES - 1:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError("SOLVE_RETRIES must be >= 1")


def _norm(s):
    return " ".join(re.sub(r"[^a-z0-9\s]", "", str(s).lower()).split())


def retrieve_pool(question, k=None):
    """Retrieve facts for one subquestion, oldest first.

    The flow chart's 'conflict resolve' is the serial rule, applied in the prompt rather than by a
    filter here: the pool is serial-ascending and ANSWER_SYS says the higher serial wins.
    """
    hits = retriever.retrieve(question, SOLVE_K if k is None else k)
    hits = sorted(hits, key=lambda h: h[0]["serial"])
    return "\n".join(f["text"] for f, _ in hits), hits


def answer_from_pool(question, pool):
    """Answer from the pool, or 'unknown' if it does not contain the answer.

    This is the flow chart's check(): the decision and the answer are one call, and 'unknown' is
    the refusal that fires the split.
    """
    raw = _chat(ANSWER_SYS, f"[Knowledge Pool]\n{pool}\n\nQuestion: {question}\nAnswer:",
                max_tokens=20)
    return raw.split("\n")[0].strip(), raw


def guess_from_pool(question, pool):
    """Commit an answer for a node nothing could repair: the closest thing in the pool rather
    than 'unknown'."""
    raw = _chat(GUESS_SYS, f"[Knowledge Pool]\n{pool}\n\nQuestion: {question}\nAnswer:",
                max_tokens=20)
    return raw.split("\n")[0].strip(), raw


def find_bridge(question, pool):
    """Name the entity that has to be resolved first. Returns (bridge_fact, raw) or (None, raw).

    Reached only after the pool has failed to answer, and it never answers the question itself, so
    it cannot overrule the reader. Its job is the evidence the split is conditioned on.
    """
    raw = _chat(BRIDGE_SYS, f"[Knowledge Pool]\n{pool}\n\nQuestion: {question}\nOutput:")
    line = next((l.strip() for l in raw.split("\n") if l.strip()), "")
    m = re.match(r"BRIDGE\s*:\s*(.+)$", line, re.I)
    if not m or _norm(m.group(1)) in ("", "none"):
        return None, raw
    return m.group(1).strip(), raw


def split(question, bridge):
    """(question, bridge fact) -> ([subq1, subq2-containing-#1], raw), or (None, raw) when the
    split is unusable: unparseable, no #1 to chain on, or subq1 just restates the parent."""
    raw = _chat(SPLIT_SYS,
                SPLIT_FEWSHOT + f"\nQuestion: {question}\nBridge fact: {bridge}\n",
                max_tokens=128)
    subqs = []
    for line in raw.split("\n"):
        m = re.match(r"^\s*[12][.)]\s*(.+)$", line.strip())
        if m:
            subqs.append(m.group(1).strip())
    if len(subqs) < 2 or "#1" not in subqs[1] or _norm(subqs[0]) == _norm(question):
        return None, raw
    return subqs[:2], raw


def _show(node):
    """Print one node: its pool, the raw LLM output at each step, and what it decided."""
    pad = "   " * node["depth"]
    print(f"{pad}[d{node['depth']}] {node['question']}")
    for p in node["pool"]:
        print(f"{pad}      [s={p['serial']:>6}] {p['text'][:100]}")
    for key in ("answer_raw", "deep_raw", "bridge_raw", "split_raw", "guess_raw"):
        if node.get(key):
            print(f"{pad}      {key:<10} {node[key][:110]!r}")
    if node["decision"] == "split":
        print(f"{pad}      bridge     {node['bridge'][:100]}")
        print(f"{pad}      -> split into: 1. {node['split'][0]}  |  2. {node['split'][1]}")
    else:
        print(f"{pad}      -> {node['decision']}: {node['answer']!r}")


def solve(question, depth, run):
    """One node of the flow chart: retrieve, try to answer, and split only if the pool cannot."""
    # A split can regenerate a subquestion an earlier hop already solved. Cached per question.
    key = _norm(question)
    if key in run["memo"]:
        cached = run["memo"][key]
        run["trace"].append({"depth": depth, "question": question, "pool": cached["pool"],
                             "decision": "reused (already solved this hop)",
                             "answer": cached["answer"]})
        run["reused"] += 1
        return cached["answer"]

    pool, hits = retrieve_pool(question)
    node = {"depth": depth, "question": question,
            "pool": [{"serial": f["serial"], "score": s, "text": f["text"]} for f, s in hits]}
    run["trace"].append(node)
    run["max_depth"] = max(run["max_depth"], depth)

    def leaf(decision, ans):
        node.update(decision=decision, answer=ans)
        run["memo"][key] = {"answer": ans, "pool": node["pool"]}
        if run["verbose"]:
            _show(node)
        return ans

    def give_up(decision):
        """No answer and no repair left: guess rather than return 'unknown'."""
        g, node["guess_raw"] = guess_from_pool(question, pool)
        run["calls"] += 1
        run["guesses"] += 1
        return leaf(decision, g)

    # No split budget left, so a refusal could not be acted on: commit in one call.
    if depth >= MAX_SPLIT_DEPTH:
        g, node["guess_raw"] = guess_from_pool(question, pool)
        run["calls"] += 1
        run["guesses"] += 1
        return leaf("guessed (depth cap)", g)

    ans, node["answer_raw"] = answer_from_pool(question, pool)
    run["calls"] += 1
    if _norm(ans) != "unknown":
        return leaf("answered", ans)

    # Optional: retry this node against a deeper pool before restructuring the question.
    # Off by default - it suppresses the split more than it helps. See the section 13 notes.
    if SOLVE_K_DEEP > SOLVE_K:
        deep_pool, deep_hits = retrieve_pool(question, k=SOLVE_K_DEEP)
        ans, node["deep_raw"] = answer_from_pool(question, deep_pool)
        run["calls"] += 1
        run["deepened"] += 1
        node["deep_pool"] = [{"serial": f["serial"], "score": sc, "text": f["text"]}
                             for f, sc in deep_hits]
        if _norm(ans) != "unknown":
            return leaf("answered (deeper pool)", ans)
        pool = deep_pool          # the bridge and the guess get the wider pool too

    bridge, node["bridge_raw"] = find_bridge(question, pool)
    run["calls"] += 1
    if bridge is None:
        return give_up("guessed (no bridge)")

    pair, node["split_raw"] = split(question, bridge)
    run["calls"] += 1
    if pair is None:
        return give_up("guessed (split failed)")

    run["splits"] += 1
    node.update(decision="split", bridge=bridge, split=pair)
    if run["verbose"]:
        _show(node)
    node["answer"] = solve_sequence(pair, depth + 1, run)   # re-enters the loop, one level deeper
    run["memo"][key] = {"answer": node["answer"], "pool": node["pool"]}
    return node["answer"]


def solve_sequence(subqs, depth, run):
    """The for-loop. Answers each subquestion in order, substituting earlier answers into later
    #k placeholders; the sequence's answer is its last answer.

    No early exit on 'unknown': every hop is attempted, because an abandoned question scores the
    same as a wrong one and abandoning loses the hops that have not run yet.
    """
    resolved = []
    for tmpl in subqs:
        resolved.append(solve(_fill(tmpl, resolved), depth, run))
    return resolved[-1] if resolved else ""


def answer_question_recursive(question, verbose=None):
    """Entry point, drop-in for section 10's answer_mh_question. Returns (answer, run); run
    carries branches / calls / splits / max_depth / branch_answers / trace."""
    verbose = SOLVE_VERBOSE if verbose is None else verbose
    subqs, n_branch = decompose_question(question)
    run = {"branches": n_branch, "calls": 1, "splits": 0, "guesses": 0, "reused": 0,
           "deepened": 0, "max_depth": 0, "memo": {}, "trace": [], "verbose": verbose}
    if verbose:
        print(f"decompose -> {n_branch} subquestion(s):")
        for s in subqs:
            print("   -", s)
    final = solve_sequence(subqs, 0, run)
    run["branch_answers"] = [n["answer"] for n in run["trace"] if n["depth"] == 0]
    run.pop("verbose")
    run.pop("memo")
    return final, run


print("ready: answer_question_recursive(q)  |  solve / solve_sequence / find_bridge / split")

ready: answer_question_recursive(q)  |  solve / solve_sequence / find_bridge / split


## 12. The arms

`retrieve` , `decompose` and `split`

In [9]:
def run_retrieve(inst):
    """One retrieval on the whole question, one answer. GUESS_SYS, not ANSWER_SYS: with no split
    to fire, 'unknown' would just be a wrong answer that scores zero."""
    _bind(build_retriever(inst))
    pool, hits = retrieve_pool(inst["question"], k=TOP_K)
    ans, raw = guess_from_pool(inst["question"], pool)
    node = {"depth": 0, "question": inst["question"], "guess_raw": raw,
            "pool": [{"serial": p["serial"], "score": s, "text": p["text"]} for p, s in hits],
            "decision": "answered (one shot)", "answer": ans}
    return ans, {"branches": 1, "calls": 1, "splits": 0, "guesses": 1, "reused": 0,
                 "deepened": 0, "max_depth": 0, "branch_answers": [ans], "trace": [node]}


def run_recursive(inst):
    """decompose -> solve_sequence. MAX_SPLIT_DEPTH is set once per arm, in run_arm."""
    _bind(build_retriever(inst))
    return answer_question_recursive(inst["question"])


ARM_FN = {"retrieve": run_retrieve, "decompose": run_recursive, "split": run_recursive}
ARM_DEPTH = {"retrieve": 0, "decompose": 0, "split": 2}
print("arms ready:", ", ".join(ARMS))

arms ready: retrieve, decompose, split


## 13. Scoring

In [10]:
def normalize_answer(s):
    s = str(s).lower()
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def _em(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))


def _f1(pred, gold):
    pt, gt = normalize_answer(pred).split(), normalize_answer(gold).split()
    if not pt or not gt:
        return float(pt == gt)
    common = collections.Counter(pt) & collections.Counter(gt)
    same = sum(common.values())
    if same == 0:
        return 0.0
    prec, rec = same / len(pt), same / len(gt)
    return 2 * prec * rec / (prec + rec)


def score(pred, inst):
    """MuSiQue scores against the answer and its aliases, best match wins."""
    golds = [inst["answer"]] + list(inst["answer_aliases"])
    return max(_em(pred, g) for g in golds), max(_f1(pred, g) for g in golds)


def sign_test(wins, losses):
    """Two-sided exact binomial on the paired wins and losses; ties carry no information."""
    n = wins + losses
    if n == 0:
        return 1.0
    tail = sum(math.comb(n, i) for i in range(min(wins, losses) + 1)) / 2 ** n
    return min(1.0, 2 * tail)


print("scoring ready")

scoring ready


## 14. Run

In [11]:
def run_arm(arm, insts, workers=None):
    """One arm over the sample. MAX_SPLIT_DEPTH is global, so it is set here, once, before any
    worker starts - never inside a worker, where the arms would race."""
    global MAX_SPLIT_DEPTH
    MAX_SPLIT_DEPTH = ARM_DEPTH[arm]
    fn = ARM_FN[arm]
    rows, traces = [None] * len(insts), [None] * len(insts)

    def one(i):
        inst = insts[i]
        try:
            ans, run = fn(inst)
        except Exception as e:
            # An API failure is not a prediction. Recorded as an error row so it can be counted
            # and re-run, never scored as a wrong answer.
            rows[i] = {"id": inst["id"], "hops": hops(inst), "pred": f"(error: {type(e).__name__})",
                       "gold": inst["answer"], "em": np.nan, "f1": np.nan, "calls": np.nan,
                       "splits": np.nan, "error": str(e)[:200]}
            return
        em, f1 = score(ans, inst)
        rows[i] = {"id": inst["id"], "hops": hops(inst), "pred": ans, "gold": inst["answer"],
                   "em": em, "f1": f1, "calls": run["calls"], "splits": run["splits"],
                   "guesses": run["guesses"], "branches": run["branches"],
                   "max_depth": run["max_depth"], "error": ""}
        traces[i] = {"id": inst["id"], "question": inst["question"], "answer": inst["answer"],
                     "pred": ans, "hops": hops(inst),
                     "trace": run["trace"], "calls": run["calls"], "splits": run["splits"]}

    with ThreadPoolExecutor(max_workers=workers or WORKERS) as ex:
        list(tqdm(ex.map(one, range(len(insts))), total=len(insts), desc=arm))

    df = pd.DataFrame(rows)
    bad = int(df["em"].isna().sum())
    if bad:
        print(f"  WARNING {bad} error rows, excluded from the score (re-run them)")
    print(f"  {arm}: EM {100*df['em'].mean():.1f}  F1 {100*df['f1'].mean():.1f}  "
          f"calls/q {df['calls'].mean():.2f}  splits/q {df['splits'].mean():.2f}")
    return df, [t for t in traces if t]


FRAMES, TRACES = {}, {}

In [12]:
ARMS = ["split"]
for _arm in ARMS:
    FRAMES[_arm], TRACES[_arm] = run_arm(_arm, QUESTIONS)

split:   0%|          | 0/2417 [00:00<?, ?it/s]

  WARNING 2417 error rows, excluded from the score (re-run them)
  split: EM nan  F1 nan  calls/q nan  splits/q nan


## 15. Results

In [ ]:
def results_table(frames, baseline="retrieve"):
    rows = []
    base = frames.get(baseline)
    for name, df in frames.items():
        ok = df.dropna(subset=["f1"])
        r = {"arm": name, "n": len(ok), "EM": 100 * ok["em"].mean(), "F1": 100 * ok["f1"].mean(),
             "calls/q": ok["calls"].mean(), "splits/q": ok["splits"].mean()}
        for h in sorted(ok["hops"].unique()):
            r[f"F1 {h}-hop"] = 100 * ok[ok["hops"] == h]["f1"].mean()
        if base is not None and name != baseline:
            m = ok.merge(base.dropna(subset=["f1"]), on="id", suffixes=("", "_b"))
            w = int((m["f1"] > m["f1_b"]).sum()); l = int((m["f1"] < m["f1_b"]).sum())
            r["vs base"] = f"+{r['F1'] - 100*m['f1_b'].mean():.1f} F1"
            r["W/L"] = f"{w}/{l}"
            r["p"] = round(sign_test(w, l), 4)
        rows.append(r)
    return pd.DataFrame(rows).set_index("arm").round(2)


results_table(FRAMES, baseline="retrieve")

In [ ]:
out = os.path.join(HERE, f"nb_rec_musique_{RETRIEVER_NAME}_{READER_MODEL}"
                         f"_n{len(QUESTIONS)}_s{SEED}_k{TOP_K}_split.json")
json.dump({"config": {"reader": READER_MODEL, "top_k": TOP_K, "solve_k": SOLVE_K,
                      "n": len(QUESTIONS), "seed": SEED, "retriever": RETRIEVER_NAME,
                      "arm_depth": ARM_DEPTH},
           "frames": {k: v.to_dict("records") for k, v in FRAMES.items()},
           "traces": TRACES},
          io.open(out, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("saved", out)

### Inspect one question

Every node the method visited: the pool it saw, the raw model output at each step, and what it
decided. This is where a wrong answer gets diagnosed - whether the gold paragraph was missing
(retrieval), present but refused (`check`), or present and misread (the reader).

In [ ]:
# Whichever of the requested arms is available -- a retrieve-only run has no "split" frame.
_arm = next((a for a in ("split", "decompose", "retrieve") if TRACES.get(a)), None)
_i = 2

if _arm is None:
    print('no traces to inspect')
else:
    print(_arm)
    _t = TRACES[_arm][_i]
    print(f"Q: {_t['question']}\ngold: {_t['answer']}   pred: {_t['pred']}   "
          f"hops={_t['hops']} calls={_t['calls']} splits={_t['splits']}\n")
    for n in _t["trace"]:
        pad = "   " * n["depth"]
        print(f"{pad}[d{n['depth']}] {n['question']}")
        for p in n.get("pool", [])[:3]:
            print(f"{pad}      [s={p['serial']:>2}] {p['text'][:110]}")
        for key in ("answer_raw", "bridge_raw", "split_raw", "guess_raw"):
            if n.get(key):
                print(f"{pad}      {key:<11}{n[key][:110]!r}")
        print(f"{pad}      -> {n.get('decision')}: {n.get('answer')!r}")